# 自定数据集
make_classification( 
                    n_samples, 
                    n_features, 
                    n_informative, 
                    n_redundant, 
                    n_repeated, 
                    n_classes, 
                    n_clusters_per_class, 
                    weights, 
                    flip_y, 
                    class_sep, 
                    hypercube, 
                    shift, 
                    scale, 
                    shuffle, 
                    random_state   
)  
n_samples: 一个整数表示将要生成的数据总量。默认值为100。
n_features: 一个整数，表示特征的数量。在make_classification中默认值为20，在make_regression中默认值为100。
n_informative: 一个整数，表示特征中比较重要的特征的数量（即可以提供更多信息量的特征数量）。在make_classification中默认值为2，在make_regression中默认值为10。
n_redundant: 一个整数，表示特征中冗余特征的数量（即不能提供更多信息量的特征数量）。在make_classification中默认值为2。
n_repeated: 一个整数，表示特征中重复特征的数量。可以模拟实际问题中因为数据提取不好造成的数据重复问题。在make_classification中默认值为0。
n_classes: 一个整数，表示分类问题中的目标类型的数量。在make_classification中默认值为2。
n_clusters_per_class：一个整数，表示分类问题中每类拥有的数据簇的数量。在make_classification中默认值为2。
weights: 浮点数的列表，表示每一类数据占总数据的比重。注意当列表中所有浮点数的值之和大于1时，可能会产生意想不到的结果。在make_classification中默认值为None。
flip_y：一个浮点数，表示噪音值。这个数越大就会使分类更困难。在make_classification中默认值为0.01。
class_sep：一个浮点数，表示类与类之间的间距。这个值越大就会使分类更容易。在make_classification中默认值为0.01。
hypercube: 一个布尔值，当为真时，表示数据簇是从超立方体（想象一下问题空间，二维问题就是正方形，三维就是立方体，更高维就是超立方体了）的顶点开始产生的。否则就表示数据簇是从随机的多平面体的顶点上生成的。说得更直白一些的话，当这个值为真时，生成的数据会更均匀一些。在make_classification中默认值为True。
shift：一个浮点数或一个长度为n_features的浮点数组或者None。表示将特征值通过某个值进行平移，不然生成的特征值就分布在0点的周围了。在make_classification中默认值为0.0。
scale：一个浮点数或一个长度为n_features的浮点数组或者None。表示将特征值与某个值相乘后的结果赋值给这个特征值，注意是先发生shift再scale，学过线性代数的同学肯定一下子就可以发现这就是对特征值做一个一维线性变换。在make_classification中默认值为1.0。
shuffle: 一个布尔值，表示是否要打乱生成的数据。在make_regression中默认为True。
random_state：None或者一个整数，当输入为一个整数时，表示这次生成数据过程的随机因子。换句话说，如果两次生成数据时，如果random_state是同一个整数，且其他参数都相同，则生成的数据是一样的。

欠采样

In [1]:
from sklearn.datasets import make_classification
# 导入make_classification函数
from imblearn.under_sampling import RandomUnderSampler
# 导入RandomUnderSampler类
from collections import Counter

# 创建一个不平衡的数据集
X, y = make_classification(n_classes=2, class_sep=2, weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0, n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)

# 打印原始数据集的类别分布
print("原始数据集类别分布：", Counter(y))

# 实例化RandomUnderSampler对象
rus = RandomUnderSampler(random_state=42)

# 对数据集进行欠采样
X_resampled, y_resampled = rus.fit_resample(X, y)

# 打印欠采样后的数据集类别分布
print("欠采样后数据集类别分布：", Counter(y_resampled))

原始数据集类别分布： Counter({np.int64(1): 900, np.int64(0): 100})
欠采样后数据集类别分布： Counter({np.int64(0): 100, np.int64(1): 100})


过采样

In [2]:
from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

# 创建一个不平衡的数据集
X, y = make_classification(n_classes=2, class_sep=2, weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0, n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)

# 打印原始数据集的类别分布
print("原始数据集类别分布：", Counter(y))

# 实例化RandomOverSampler对象
ros = RandomOverSampler(random_state=42)

# 对数据集进行过采样
X_resampled, y_resampled = ros.fit_resample(X, y)

# 打印过采样后的数据集类别分布
print("过采样后数据集类别分布：", Counter(y_resampled))

原始数据集类别分布： Counter({np.int64(1): 900, np.int64(0): 100})
过采样后数据集类别分布： Counter({np.int64(0): 900, np.int64(1): 900})


阈值移动

In [3]:
#分类的时候，当不同类别的样本量差异很大时，很容易影响分类结果，因此要么每个类别的数据量大致相同，要么就要进行校正。
#sklearn的做法可以是加权，加权就要涉及到class_weight和sample_weight
#当不设置class_weight参数时，默认值是所有类别的权值为1

#那么'balanced'的计算方法是什么呢？
import numpy as np

y = [0,0,0,0,0,0,0,0,1,1,1,1,1,1,2,2]  #标签值，一共16个样本

a = np.bincount(y)  # array([8, 6, 2], dtype=int64) 计算每个类别的样本数量
aa = 1/a  #倒数 array([0.125     , 0.16666667, 0.5       ])
print(aa)


[0.125      0.16666667 0.5       ]


In [19]:
from sklearn.utils.class_weight import compute_class_weight 
y = [0,0,0,0,0,0,0,0,1,1,1,1,1,1,2,2]

# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
print("类别权重：", class_weights) # [0.66666667 0.88888889 2.66666667]


#weight_ = n_samples / (n_classes * np.bincount(y))
weight_ = 16 / (3 * np.bincount(y))
print(16/(3*8))  #输出 0.6666666666666666
print(16/(3*6))  #输出 0.8888888888888888
print(16/(3*2))  #输出 2.6666666666666665
weight_

类别权重： [0.66666667 0.88888889 2.66666667]
0.6666666666666666
0.8888888888888888
2.6666666666666665


array([0.66666667, 0.88888889, 2.66666667])

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 创建一个不平衡的数据集
X, y = make_classification(n_classes=2, class_sep=2, weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0, n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 创建逻辑回归模型，并设置class_weight为'balanced'
# class_weight 也可以设置为 {0:99:，1:1}
logreg = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000)
# 或者logreg = LogisticRegression(class_weight={0:9, 1:1}, solver='liblinear', max_iter=1000)
# 训练模型
logreg.fit(X_train, y_train)

# 预测测试集
y_pred = logreg.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print("逻辑回归模型的准确率：", accuracy)

逻辑回归模型的准确率： 0.99


In [18]:
# 未使用阈值移动
logreg1 = LogisticRegression(solver='liblinear', max_iter=1000)
# logreg = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000)
# 训练模型
logreg1.fit(X_train, y_train)

# 预测测试集
y_pred1 = logreg1.predict(X_test)

# 计算准确率
accuracy1 = accuracy_score(y_test, y_pred1)
print("逻辑回归模型的准确率：", accuracy1)

逻辑回归模型的准确率： 0.985
